# 04 — Classification: Prediksi Pelanggan High Value (Decision Tree)

Notebook ini melatih Decision Tree untuk memprediksi apakah seorang pelanggan termasuk kategori high value berdasarkan usia, kota, dan perilaku belanja.

## 1. Setup

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

import sys
sys.path.insert(0, "/home/jovyan/work")

from analysis.spark_session import create_spark_session
spark = create_spark_session("04 - Classification")

## 2. Baca Data

In [ ]:
BUCKET = "datalake"
df_features = spark.read.csv(f"s3a://{BUCKET}/processed/customer_features/", header=True, inferSchema=True)

print("Distribusi label is_high_value:")
df_features.groupBy("is_high_value").count().show()
df_features.show(5)

## 3. Latih Decision Tree

In [ ]:
from analysis.mining.classification import run_decision_tree

df_predictions, auc = run_decision_tree(
    df_features,
    numeric_cols=["age", "total_orders", "total_spend", "avg_spend_per_order"],
    categorical_cols=["city"],
    label_col="is_high_value",
    test_ratio=0.2,
)

print(f"
Area Under ROC (AUC): {auc:.3f}")
print("
Hasil prediksi:")
df_predictions.select("customer_name", "city", "total_spend", "is_high_value", "prediction").show()

## 4. Evaluasi Model

In [ ]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

evaluator = MulticlassClassificationEvaluator(labelCol="is_high_value", predictionCol="prediction")

accuracy  = evaluator.setMetricName("accuracy").evaluate(df_predictions)
f1        = evaluator.setMetricName("f1").evaluate(df_predictions)
precision = evaluator.setMetricName("weightedPrecision").evaluate(df_predictions)
recall    = evaluator.setMetricName("weightedRecall").evaluate(df_predictions)

print(f"Accuracy  : {accuracy:.3f}")
print(f"F1 Score  : {f1:.3f}")
print(f"Precision : {precision:.3f}")
print(f"Recall    : {recall:.3f}")
print(f"AUC       : {auc:.3f}")

## 5. Confusion Matrix

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pyspark.sql import functions as F

# Hitung confusion matrix
cm = df_predictions.groupBy("is_high_value", "prediction").count().toPandas()
matrix = cm.pivot(index="is_high_value", columns="prediction", values="count").fillna(0)

plt.figure(figsize=(6, 4))
sns.heatmap(matrix, annot=True, fmt=".0f", cmap="Blues",
            xticklabels=["Prediksi: 0","Prediksi: 1"],
            yticklabels=["Aktual: 0","Aktual: 1"])
plt.title("Confusion Matrix")
plt.tight_layout()
plt.show()

## 6. Simpan Hasil ke MinIO

In [ ]:
df_predictions.write.mode("overwrite").option("header", True) \n    .csv(f"s3a://{BUCKET}/processed/classification_results/")
print("Hasil klasifikasi disimpan ke processed zone.")